In [1]:
library(tidyverse)
library(tis)
library(baseballr)
library(hoopR)
library(MASS)
library(glmnet)
library(mpath)
library(glmmTMB)
library(Matrix)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘tis’


The following objects are masked from ‘package:lubridate’:

    day, hms, month, period, POSIXct, quarter, today, year, ymd


The following object is masked from ‘package:dplyr’:

    between



Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glm

In this section, let's build all of the models that we trained. Of note, I have not tested any of these models on the testing set yet, so there is no data leakage problem. In other words, they're pre-registered.

In [2]:
setwd('..')
getwd()

df <- read_csv("./data/final-dataset.csv")

# Split into Training and Testing sets. Treat hour as a factor.
train <- df %>% filter(year(date) < 2025) %>% mutate(hour = as_factor(hour))
test <- df %>% filter(year(date) == 2025) %>% mutate(hour = as_factor(hour))

# For Ridge Poisson, Ridge NB and Zero-Inflated NB GLM
# Remove the intercept since glmnet adds it automatically
model_formula <- as.formula("ridership ~ hour + destination + day + is_holiday + is_giants_home + is_as_home + warriors_at_chase + season - 1")
X_train <- model.matrix(model_formula, data = train)
y_train <- train$ridership

X_test <- model.matrix(model_formula, data = test)
y_test <- test$ridership

best_lambda_poisson <- 7.08165174440212
best_lambda_nb <- 0.5994843
theta_nb <- 3.18501460012295

[1] "/accounts/masters/ben_khothsombath/repos/230A-final-project"

Rows: 1130841 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): destination, day, season
dbl  (2): hour, ridership
lgl  (9): destination_green, destination_red, destination_yellow, destinatio...
date (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Some notes for these models...

* The log-transformed LM needs to have its ridership converted back to the original scale in order for model performance to be comparible. However, it is not enough to just exponentiate the predicted values, since the expectation of a log is not the same as the log of the expectation. Thus, we need to perform a correction. Multiplying $\exp(\hat{y})$ by $\exp(\hat{\sigma}^2/2)$ will make this estimate unbiased, but only if the residuals were normally distributed in the log space.
* For the 5-fold CV, we used time-based splits. Technically, when doing a time-based split, you shouldn't test a past fold using data trained from a future fold (for example, fold 3 will be trained using folds 1,2,4,5) where 4 and 5 are from the future. However, we're ignoring the inherent? (we'd want to show that autocorrelation exists) time-dependence and are going to assume that it's fine. This can be part of the discussion section.
* For the selection of lambda for ridge NB, we ran into some memory issues, so we opted instead to just use a validation set. We reserve 2023 as the inner training set and 2024 as the inner validation set and then retrain the full model on both.
* I was originally thinking of combining both zero-inflated NB and ridge NB, but the package that can implement both ran into some memory errors, so we'll just separate it out for now.
* For ridge runs, we don't need to standardize because all variables are indicators already.

In [3]:
# Basic Linear Model
basic_lm <- lm(ridership~., data=train[, -c(1)])

# Log-Transformed LM
log_lm <- lm(log(ridership) ~ ., data=train[, -c(1)])

# Poisson GLM
poisson_lm <- glm(ridership ~ ., data = train[, -c(1)], family = poisson(link = "log"))

# Negative Binomial Model
nb_lm <- glm.nb(ridership ~ ., data = train[, -c(1)])

# Poisson GLM with Ridge selected by 5-fold CV
poisson_ridge <- glmnet(
    x = X_train,
    y = y_train,
    family = "poisson",
    alpha = 0, # 0 for ridge, 1 for LASSO
    lambda = best_lambda_poisson
)

# NB GLM with Ridge - selected using validation set
nb_ridge <- glmnet(
    x = X_train,
    y = y_train,
    family = negative.binomial(theta = theta_nb),
    alpha = 0,
    lambda = best_lambda_nb
)

# Zero-inflated NB
model_zinb <- glmmTMB(
    ridership ~ hour + destination + day + is_holiday + is_giants_home + is_as_home + warriors_at_chase + season,
    data = train,
    ziformula = ~1,
    family = nbinom2 # nbinom2 = NB with quadratic variance (same as glm.nb)
)

Warning message in finalizeTMB(TMBStruc, obj, fit, h, data.tmb.old):
“Model convergence problem; iteration limit reached without convergence (10). See vignette('troubleshooting'), help('diagnose')”


Now, let's use these models to predict the test error.

In [4]:
# Helper to compute MSE and RMSE cleanly
eval_preds <- function(preds, actual) {
  mse  <- mean((actual - preds)^2)
  rmse <- sqrt(mse)
  list(mse = mse, rmse = rmse)
}

# Model 1: Basic LM
preds_lm   <- predict(basic_lm, newdata = test)
# Clip negatives: LM has no non-negativity constraint, but ridership can't be negative
preds_lm   <- pmax(preds_lm, 0)
result_lm  <- eval_preds(preds_lm, y_test)

# Model 2: Log-transformed LM
# E[Y] = exp(mu + sigma^2/2) when residuals are normal in log space.
# Correction factor exp(sigma^2/2) makes the back-transformed prediction unbiased.
# We should check the residual plot of this model to check the appropriateness of this correction factor.
sigma2_log_lm <- var(residuals(log_lm))
preds_log_lm  <- exp(predict(log_lm, newdata = test)) * exp(sigma2_log_lm / 2)
result_log_lm <- eval_preds(preds_log_lm, y_test)

# Model 3: Poisson GLM
preds_poisson  <- predict(poisson_lm, newdata = test, type = "response")
result_poisson <- eval_preds(preds_poisson, y_test)

# Model 4: NB GLM
preds_nb  <- predict(nb_lm, newdata = test, type = "response")
result_nb <- eval_preds(preds_nb, y_test)

# Model 5: Ridge Poisson
preds_poisson_ridge  <- as.numeric(predict(poisson_ridge, newx = X_test, type = "response"))
result_poisson_ridge <- eval_preds(preds_poisson_ridge, y_test)

# Model 6: Ridge NB
preds_nb_ridge  <- as.numeric(predict(nb_ridge, newx = X_test, type = "response"))
result_nb_ridge <- eval_preds(preds_nb_ridge, y_test)

# Model 7: Zero-Inflated NB
preds_zinb  <- predict(model_zinb, newdata = test, type = "response")
result_zinb <- eval_preds(preds_zinb, y_test)

# Print Results
results <- tibble(
  Model = c(
    "Basic LM",
    "Log-Transformed LM (bias-corrected)",
    "Poisson GLM",
    "Negative Binomial GLM",
    "Ridge Poisson (lambda = 7.082)",
    "Ridge NB (lambda = 0.599)",
    "Zero-Inflated NB"
  ),
  Lambda_Selection = c(
    "—", "—", "—", "—",
    "5-fold time-based CV",
    "Validation set (2023 train / 2024 val)",
    "—"
  ),
  Test_MSE  = round(c(
    result_lm$mse, result_log_lm$mse, result_poisson$mse,
    result_nb$mse, result_poisson_ridge$mse,
    result_nb_ridge$mse, result_zinb$mse
  ), 2),
  Test_RMSE = round(c(
    result_lm$rmse, result_log_lm$rmse, result_poisson$rmse,
    result_nb$rmse, result_poisson_ridge$rmse,
    result_nb_ridge$rmse, result_zinb$rmse
  ), 2)
)

print(results)

# A tibble: 7 × 4
  Model                               Lambda_Selection        Test_MSE Test_RMSE
  <chr>                               <chr>                      <dbl>     <dbl>
1 Basic LM                            —                         41677.      204.
2 Log-Transformed LM (bias-corrected) —                         47682.      218.
3 Poisson GLM                         —                         32969.      182.
4 Negative Binomial GLM               —                         46721.      216.
5 Ridge Poisson (lambda = 7.082)      5-fold time-based CV      33584.      183.
6 Ridge NB (lambda = 0.599)           Validation set (2023 t…   43978.      210.
7 Zero-Inflated NB                    —                         46697.      216.
